# Klasifikasi Sentimen Teks Bahasa Indonesia

**Notebook presentasi — Proyek UAS Machine Learning**

Notebook ini mendemonstrasikan cara aplikasi mengubah teks ulasan menjadi prediksi sentimen **positive**, **negative**, atau **neutral**.

### Tujuan
- Menjelaskan tahapan preprocessing dan representasi teks dengan TF-IDF.
- Menggunakan model terbaik yang sudah dilatih, tanpa melatih ulang.
- Menunjukkan prediksi teks, pemrosesan CSV, dan hasil evaluasi.

**Alur:** ulasan → preprocessing → TF-IDF → klasifikasi → label dan skor → evaluasi.

> Jalankan sel kode dari atas ke bawah. Pastikan notebook dibuka dari root folder proyek dan file `model/model.joblib` tersedia.

## 1. Persiapan dan memuat artefak

Model disimpan sebagai pipeline scikit-learn yang berisi TF-IDF dan classifier. Metrik evaluasi dibaca dari JSON yang dihasilkan saat pelatihan, sehingga notebook ini tidak mengubah model.

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "preprocessing.py").exists():
    raise FileNotFoundError(
        "Buka notebook dari root proyek yang berisi preprocessing.py, data/, dan model/."
    )

from preprocessing import clean_text

MODEL_PATH = ROOT_DIR / "model" / "model.joblib"
METRICS_PATH = ROOT_DIR / "model" / "metrics.json"

model = joblib.load(MODEL_PATH)
with METRICS_PATH.open("r", encoding="utf-8") as file:
    metrics = json.load(file)

print(f"Model terbaik: {metrics['best_model']}")
print(f"Kelas sentimen: {', '.join(metrics['labels'])}")

Model terbaik: Linear SVM
Kelas sentimen: negative, neutral, positive


## 2. Preprocessing teks

Fungsi `clean_text()` dari proyek melakukan normalisasi huruf menjadi lowercase, menghapus URL dan karakter yang bukan huruf, menormalkan beberapa slang, lalu menghapus stopword. Kata negasi seperti **tidak** sengaja dipertahankan karena dapat mengubah sentimen.

Contoh berikut memperlihatkan teks asli dan bentuk yang akan diproses model.

In [2]:
contoh_asli = "Gue kecewa banget! Pelayanannya gak ramah... https://contoh.id"
print("Teks asli       :", contoh_asli)
print("Setelah dibersihkan:", clean_text(contoh_asli))

Teks asli       : Gue kecewa banget! Pelayanannya gak ramah... https://contoh.id
Setelah dibersihkan: gue kecewa banget pelayanannya tidak ramah


## 3. Dari kata ke prediksi

TF-IDF memberi bobot pada kata dan pasangan kata (unigram dan bigram). Model terpilih adalah **Linear SVM**. Karena classifier SVM pada proyek ini menyediakan `decision_function`, bukan `predict_proba`, skor keputusannya diubah dengan softmax agar dapat dibandingkan sebagai skor relatif.

> **Catatan interpretasi:** softmax dari skor SVM bukan probabilitas terkalibrasi. Nilai “keyakinan” di bawah sebaiknya dibaca sebagai skor relatif antarkelas, bukan jaminan bahwa prediksi benar dengan peluang tersebut.

In [3]:
def softmax_from_logits(logits):
    """Ubah skor keputusan menjadi skor relatif yang jumlahnya 1 per teks."""
    logits = np.asarray(logits)
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=1, keepdims=True)


def predict_texts(texts):
    """Bersihkan beberapa teks, lalu kembalikan kelas dan skor tiap kelas."""
    cleaned_texts = [clean_text(text) for text in texts]
    classifier = model.named_steps["clf"]
    vectorizer = model.named_steps["tfidf"]
    features = vectorizer.transform(cleaned_texts)

    if hasattr(classifier, "predict_proba"):
        scores = classifier.predict_proba(features)
    else:
        scores = softmax_from_logits(classifier.decision_function(features))

    return classifier.classes_, scores


def classify_csv(path):
    """Klasifikasikan CSV yang memiliki kolom teks bernama 'teks'."""
    dataframe = pd.read_csv(path)
    if "teks" not in dataframe.columns:
        raise ValueError("File CSV harus memiliki kolom 'teks'.")

    classes, scores = predict_texts(dataframe["teks"].fillna("").tolist())
    best_indices = scores.argmax(axis=1)
    dataframe["prediksi"] = classes[best_indices]
    dataframe["keyakinan_%"] = (scores.max(axis=1) * 100).round(1)
    return dataframe

## 4. Demo klasifikasi teks

Masukkan tiga contoh dengan nuansa berbeda. Tabel menampilkan prediksi utama dan skor relatif untuk setiap kelas; grafik batang teks menyajikan nilai yang sama agar mudah dibahas saat presentasi.

In [4]:
contoh_teks = [
    "Makanan di restoran ini enak banget, pelayanan ramah dan tempat nyaman. Pasti balik lagi.",
    "Saya kecewa dengan pelayanan yang buruk. Makanannya tidak enak dan tempatnya kotor.",
    "Kemarin saya pergi ke mal dekat rumah. Tempatnya cukup luas dan banyak toko.",
]

classes, scores = predict_texts(contoh_teks)
hasil = pd.DataFrame(scores, columns=classes, index=["Contoh positif", "Contoh negatif", "Contoh netral"])
hasil.insert(0, "teks", contoh_teks)
hasil.insert(1, "prediksi", np.asarray(classes)[scores.argmax(axis=1)])
hasil.insert(2, "skor_tertinggi_%", (scores.max(axis=1) * 100).round(1))
display(hasil)

display(pd.DataFrame(scores, columns=classes, index=hasil.index).style.format("{:.1%}").bar(vmin=0, vmax=1))

,teks,prediksi,skor_tertinggi_%,negative,neutral,positive
Contoh positif,"Makanan di restoran ini enak banget, pelayanan...",positive,70.4,0.169499,0.126069,0.704432
Contoh negatif,Saya kecewa dengan pelayanan yang buruk. Makan...,negative,78.2,0.781688,0.093711,0.124601
Contoh netral,Kemarin saya pergi ke mal dekat rumah. Tempatn...,neutral,53.0,0.281511,0.529593,0.188896


,negative,neutral,positive
Contoh positif,16.9%,12.6%,70.4%
Contoh negatif,78.2%,9.4%,12.5%
Contoh netral,28.2%,53.0%,18.9%


## 5. Klasifikasi banyak teks dari CSV

Aplikasi Streamlit menyediakan unggah CSV. Dalam notebook ini, fungsi `classify_csv()` menerima **path** ke CSV yang memiliki kolom `teks`, lalu menambahkan kolom `prediksi` dan `keyakinan_%`. Contoh berikut menggunakan sebagian data proyek untuk demonstrasi; ganti path dengan CSV lain sesuai kebutuhan.

In [5]:
CSV_PATH = ROOT_DIR / "data" / "dataset_ulasan.csv"
hasil_csv = classify_csv(CSV_PATH).head(10)
display(hasil_csv)

,teks,label,prediksi,keyakinan_%
0,kemarin gue datang ke tempat makan baru yang a...,negative,negative,65.9
1,kayak nya sih gue tidak akan mau balik lagi ke...,negative,negative,78.0
2,"kalau dipikir-pikir , sebenarnya tidak ada yan...",negative,negative,74.9
3,ini pertama kalinya gua ke bank buat ngurusin ...,negative,negative,70.2
4,waktu sampai dengan gue pernah disuruh ibu lat...,negative,negative,59.2
5,pelayanan di hotel salak bogor tidak sebagus y...,negative,negative,71.1
6,ada apa dengan young lex ? kenapa dia harus me...,negative,negative,79.5
7,"gue pesimis sama manusia , terutama dalam ngur...",negative,negative,74.2
8,saya kecewa karena pengeditan biodata penumpan...,negative,negative,75.0
9,saya hari ini melakukan pemesanan dan tiket sa...,negative,negative,76.2


## 6. Evaluasi model

Model dibandingkan menggunakan cross-validation dan data uji. **Precision** mengukur ketepatan prediksi per kelas, **recall** mengukur banyaknya contoh kelas yang berhasil ditemukan, dan **macro-F1** merata-ratakan F1 antarkelas sehingga tiap kelas mendapat bobot setara. Model terbaik proyek dipilih berdasarkan macro-F1.

In [6]:
perbandingan = pd.DataFrame(metrics["results"]).set_index("model")
display(perbandingan.style.format("{:.3f}"))

print(
    f"Data latih: {metrics['n_train']} | Data uji: {metrics['n_test']} "
    f"| Model terbaik: {metrics['best_model']}"
)

,cv_acc,test_acc,precision,recall,f1
model,,,,,
Multinomial Naive Bayes,0.735,0.760,0.836,0.649,0.639
Linear SVM,0.825,0.790,0.746,0.736,0.740
Logistic Regression,0.805,0.770,0.728,0.720,0.723


Data latih: 400 | Data uji: 100 | Model terbaik: Linear SVM


### Confusion matrix

Baris menunjukkan label aktual dan kolom menunjukkan hasil prediksi. Nilai pada diagonal adalah prediksi yang tepat; nilai di luar diagonal menunjukkan kelas yang tertukar.

In [7]:
confusion = pd.DataFrame(
    metrics["confusion_matrix"],
    index=[f"Aktual {label}" for label in metrics["labels"]],
    columns=[f"Prediksi {label}" for label in metrics["labels"]],
)
display(confusion)

,Prediksi negative,Prediksi neutral,Prediksi positive
Aktual negative,35,3,3
Aktual neutral,8,9,1
Aktual positive,2,4,35


## 7. Kesimpulan untuk presentasi

- Sistem membersihkan ulasan, mengubah teks menjadi fitur TF-IDF unigram/bigram, lalu memprediksi sentimen.
- Dari model yang dibandingkan, **Linear SVM** dipilih sebagai model terbaik.
- Pada data uji proyek, Linear SVM memperoleh **akurasi 0,790** dan **macro-F1 0,740** (pembulatan dari metrik tersimpan).
- Sistem mendukung prediksi satu atau banyak teks, tetapi tetap dapat keliru—terutama saat bahasa ambigu, sarkasme, atau kosakata tidak terwakili data latih.
- Skor relatif yang berasal dari softmax skor keputusan SVM bukan probabilitas terkalibrasi.

### Menjalankan notebook
Buka notebook ini dari root proyek, pilih kernel Python yang memiliki dependensi pada `requirements.txt`, lalu jalankan **Run All**. Model dan metrik harus sudah tersedia di folder `model/`; untuk aplikasi web interaktif, jalankan `streamlit run app.py` dari terminal.